# Day 12, Step 2 — Context Selection (agent)

**Problem:** `example.py` always sends a fixed `USER` + `FACT` block — there's never a question of whether the fact belongs. In a real system you usually have more candidate context than you should send: several retrieved passages, each possibly relevant or not. Sending all of it wastes budget and can bury the part the model actually needs.

This notebook is one of Day 12's four notebooks that deliberately cross this course's agent boundary (see `CLAUDE.md`'s "Model-backed days" section): instead of a hand-written keyword rule, the **model itself** decides which retrieved passage is relevant — and LangGraph routes to a different node depending on that decision, the same pattern taught in Day 6.

**Graph:** `START -> classify_rag -> (agent decision) -> include_rag / skip_rag -> build_context -> ask_model -> END`

## Setup

In [1]:
from typing import TypedDict

import ollama
from langgraph.graph import END, START, StateGraph

MODEL = "gpt-oss:120b-cloud"


## State

`rag_choice` holds the model's own decision (a candidate's label, or `NONE`) — the field the routing function reads to pick the next node.

In [2]:
class State(TypedDict):
    request: str
    rag_candidates: list[tuple[str, str]]
    rag_choice: str
    selected_rag: list[tuple[str, str]]
    context: str
    answer: str


## `classify_rag` — the agent decision

Shows the model every candidate passage and asks it to pick the one relevant label, or `NONE`. The allowed answers are a fixed list (this session's candidate labels plus `NONE`) — exactly Day 6's "one action from a fixed allowlist" pattern. Anything outside that list is corrected to `NONE`, so an unparseable reply can never route anywhere unexpected.

In [3]:
def classify_rag(state: State):
    labels = [label for label, _ in state["rag_candidates"]]
    listing = "\n".join(f"{label}: {text}" for label, text in state["rag_candidates"])
    prompt = (
        "Which of these knowledge base passages, if any, is relevant to the question?\n"
        f"Return only one label from this list, or NONE: {', '.join(labels)}\n\n"
        f"Question: {state['request']}\n\n{listing}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    choice = response["message"]["content"].strip().upper()
    valid = {label.upper() for label in labels} | {"NONE"}
    if choice not in valid:
        choice = "NONE"
    return {"rag_choice": choice}


## `route_after_classify` — where the agent boundary lives

This is the function LangGraph uses to pick the next node. Because `rag_choice` came from the model's own output, the model is the thing actually choosing whether `include_rag` or `skip_rag` runs — that's the agent boundary, not the `ollama.chat` call by itself.

In [4]:
def route_after_classify(state: State):
    if state["rag_choice"] == "NONE":
        return "skip_rag"
    return "include_rag"


## `include_rag` and `skip_rag` — the two possible next nodes

`include_rag` looks up the candidate the model chose and keeps only that one; `skip_rag` keeps none. Either way, execution continues to `build_context` next.

In [5]:
def include_rag(state: State):
    for label, text in state["rag_candidates"]:
        if label.upper() == state["rag_choice"]:
            return {"selected_rag": [(label, text)]}
    return {"selected_rag": []}


def skip_rag(state: State):
    return {"selected_rag": []}


## `build_context` and `ask_model`

In [6]:
def build_context(state: State):
    lines = [f"TASK: {state['request']}"]
    for label, text in state["selected_rag"]:
        lines.append(f"RAG[{label}]: {text}")
    return {"context": "\n".join(lines)}


def ask_model(state: State):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": state['context']}],
    )
    return {"answer": response["message"]["content"]}


## Build the graph

`classify_rag` routes conditionally via `add_conditional_edges` to either `include_rag` or `skip_rag`; both rejoin at `build_context`.

In [7]:
graph_builder = StateGraph(State)
graph_builder.add_node("classify_rag", classify_rag)
graph_builder.add_node("include_rag", include_rag)
graph_builder.add_node("skip_rag", skip_rag)
graph_builder.add_node("build_context", build_context)
graph_builder.add_node("ask_model", ask_model)

graph_builder.add_edge(START, "classify_rag")
graph_builder.add_conditional_edges("classify_rag", route_after_classify)
graph_builder.add_edge("include_rag", "build_context")
graph_builder.add_edge("skip_rag", "build_context")
graph_builder.add_edge("build_context", "ask_model")
graph_builder.add_edge("ask_model", END)
graph = graph_builder.compile()


## Run it

One candidate (KB-102) is about password resets, the other (KB-205) is about the cafeteria menu. Watch `rag_choice` in the printed context — an on-topic model should pick `KB-102` and route through `include_rag`, leaving the cafeteria passage out entirely.

In [ ]:
result = graph.invoke(
    {
        "request": "My laptop screen is broken?",
        "rag_candidates": [
            ("KB-102", "Use /reset to reset your password."),
            ("KB-205", "The cafeteria menu changes every Monday."),
        ],
    }
)
print("Context sent to the model:\n", result["context"])
print("\nAnswer:\n", result["answer"])


Context sent to the model:
 TASK: On which day does the cafeteria menu changes?
RAG[KB-205]: The cafeteria menu changes every Monday.

Answer:
 The cafeteria menu changes **every Monday**.
